# Exibindo as tabelas dentro do volume

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/projeto/data_schema/data_volume"))

# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Criando o database bronze

In [0]:
%sql
USE CATALOG projeto;
-- DROP DATABASE bronze CASCADE; -- Executar se tiver uma bronze já criada
CREATE DATABASE IF NOT EXISTS bronze;

# Lendo os arquivos csv para a camada bronze

In [0]:
# Base path inside your Unity Catalog volume
base_path = "dbfs:/Volumes/projeto/data_schema/data_volume/"

# Mapping of file names to Bronze table names
files_to_tables = {
    "Chamados_Hora.csv": "bronze.chamados_hora",
    "base_atendentes.csv": "bronze.base_atendentes",
    "base_motivos.csv": "bronze.base_motivos",
    "canais.csv": "bronze.canais",
    "chamados.csv": "bronze.chamados",
    "clientes.csv": "bronze.clientes",
    "custos.csv": "bronze.custos",
    "pesquisa_satisfacao.csv": "bronze.pesquisa_satisfacao"
}


for file_name, table_name in files_to_tables.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}{file_name}")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f" Table created: {table_name}")   

# Exibindo as tabelas criadas

In [0]:
%sql
SHOW TABLES IN bronze;